In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, jaccard_score
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
base_path = '/content/drive/My Drive/'
image_dir = os.path.join(base_path, 'images')
annotation_dir = os.path.join(base_path, 'annotations')
train_file = os.path.join(base_path, 'train.txt')
train_cls_file = os.path.join(base_path, 'train_cls.txt')
val_file = os.path.join(base_path, 'val.txt')
val_cls_file = os.path.join(base_path, 'val_cls.txt')
checkpoint_dir = os.path.join(base_path, 'checkpoints')


In [ ]:
def load_data(file_list, image_dir, annotation_dir, num_classes=8, target_size=(128, 128)):
    images = []
    masks = []
    with open(file_list, 'r') as f:
        lines = f.readlines()
        for line in lines:
            image_file = os.path.join(image_dir, f"{line.strip()}.JPG")
            mask_file = os.path.join(annotation_dir, f"{line.strip()}.PNG")
            image = img_to_array(load_img(image_file, target_size=target_size)) / 255.0
            mask = img_to_array(load_img(mask_file, target_size=target_size, color_mode="grayscale"))
            mask = mask.squeeze()  # remove the single channel dimension
            mask = np.clip(mask, 0, num_classes - 1)  # ensure mask values are within the expected range
            mask = to_categorical(mask, num_classes=num_classes)  # multi-class segmentation
            images.append(image)
            masks.append(mask)
    return np.array(images), np.array(masks)

# Load training and validation data
train_images, train_masks = load_data(train_file, image_dir, annotation_dir, num_classes=8)
val_images, val_masks = load_data(val_file, image_dir, annotation_dir, num_classes=8)


In [ ]:
def unet_model(input_size=(128, 128, 3), num_classes=8):
    inputs = Input(input_size)

    c1 = Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = MaxPooling2D((2, 2))(c4)

    c5 = Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)

    u6 = UpSampling2D((2, 2))(c5)
    u6 = Concatenate()([u6, c4])
    c6 = Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    c6 = Conv2D(512, (3, 3), activation='relu', padding='same')(c6)

    u7 = UpSampling2D((2, 2))(c6)
    u7 = Concatenate()([u7, c3])
    c7 = Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    c7 = Conv2D(256, (3, 3), activation='relu', padding='same')(c7)

    u8 = UpSampling2D((2, 2))(c7)
    u8 = Concatenate()([u8, c2])
    c8 = Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    c8 = Conv2D(128, (3, 3), activation='relu', padding='same')(c8)

    u9 = UpSampling2D((2, 2))(c8)
    u9 = Concatenate()([u9, c1])
    c9 = Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    c9 = Conv2D(64, (3, 3), activation='relu', padding='same')(c9)

    outputs = Conv2D(num_classes, (1, 1), activation='softmax')(c9)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model


# model = unet_model(num_classes=8)
# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# Get list of checkpoint files
checkpoint_files = sorted([os.path.join(checkpoint_dir, f) for f in os.listdir(checkpoint_dir) if f.endswith('.h5')])

# Load the most recent checkpoint
latest_checkpoint = checkpoint_files[-1]
print(f"Loading model from {latest_checkpoint}")
model = load_model(latest_checkpoint)


Loading model from /content/drive/My Drive/checkpoints/epoch_40.h5


In [ ]:
# Define callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="/content/drive/My Drive/checkpoints/epoch_{epoch:02d}.h5",
        save_best_only=False,  # Save the model for every epoch
        save_freq='epoch',
        monitor="val_loss"
    ),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, verbose=1)
]

# Ensure the checkpoint directory exists
os.makedirs("/content/drive/My Drive/checkpoints", exist_ok=True)

# Resume training the model
initial_epoch = int(latest_checkpoint.split('_')[-1].split('.')[0])  # Extract the epoch number from the checkpoint filename
history = model.fit(
    train_images, train_masks,
    validation_data=(val_images, val_masks),
    epochs=40,
    batch_size=32,  # Batch size set to 32
    callbacks=callbacks,
    initial_epoch=initial_epoch  # Start from the last completed epoch
)

# Evaluate the model on the validation set
loss, accuracy = model.evaluate(val_images, val_masks)
print(f"Validation Loss: {loss}")
print(f"Validation Accuracy: {accuracy}")


Epoch 36/40
32/32 [==============================] - ETA: 0s - loss: 0.0320 - accuracy: 0.9863  

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


32/32 [==============================] - 2492s 78s/step - loss: 0.0320 - accuracy: 0.9863 - val_loss: 0.2282 - val_accuracy: 0.9457
Epoch 37/40
32/32 [==============================] - 2466s 77s/step - loss: 0.0324 - accuracy: 0.9863 - val_loss: 0.2601 - val_accuracy: 0.9470
Epoch 38/40
32/32 [==============================] - 2480s 78s/step - loss: 0.0295 - accuracy: 0.9874 - val_loss: 0.2633 - val_accuracy: 0.9478
Epoch 39/40
32/32 [==============================] - 2470s 77s/step - loss: 0.0269 - accuracy: 0.9884 - val_loss: 0.2773 - val_accuracy: 0.9456
Epoch 40/40
15/15 [==============================] - 285s 19s/step - loss: 0.2652 - accuracy: 0.9466
Validation Loss: 0.26518914103507996
Validation Accuracy: 0.946648359298706


In [ ]:
# Get the predicted masks for the validation set
pred_masks = model.predict(val_images)
pred_masks = np.argmax(pred_masks, axis=-1)
true_masks = np.argmax(val_masks, axis=-1)

# Flatten the arrays for metric calculations
pred_masks_flat = pred_masks.flatten()
true_masks_flat = true_masks.flatten()

# Compute classification metrics
accuracy = accuracy_score(true_masks_flat, pred_masks_flat)
precision = precision_score(true_masks_flat, pred_masks_flat, average='weighted')
recall = recall_score(true_masks_flat, pred_masks_flat, average='weighted')
f1 = f1_score(true_masks_flat, pred_masks_flat, average='weighted')
iou = jaccard_score(true_masks_flat, pred_masks_flat, average='weighted')

print(f"Overall Accuracy: {accuracy}")
print(f"Overall Precision: {precision}")
print(f"Overall Recall: {recall}")
print(f"Overall F1 Score: {f1}")
print(f"Overall IoU Score: {iou}")




15/15 ━━━━━━━━━━━━━━━━━━━━ 288s 19s/step
Overall Accuracy: 0.946648376074427
Overall Precision: 0.9457402732154077
Overall Recall: 0.946648376074427
Overall F1 Score: 0.9461073563692033
Overall IoU Score: 0.9022947648137637


In [ ]:
# Initialize lists to hold accuracy and loss data
train_acc = []
val_acc = []
train_loss = []
val_loss = []

# Loop through each checkpoint to gather accuracy and loss data
for epoch in range(1, 41):
    checkpoint_file = os.path.join(checkpoint_dir, f'epoch_{epoch:02d}.h5')
    if os.path.exists(checkpoint_file):
        model = load_model(checkpoint_file, compile=False)
        model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        sample_images = train_images[:100]
        sample_masks = train_masks[:100]
        history = model.evaluate(train_images, train_masks, verbose=0)
        train_loss.append(history[0])
        train_acc.append(history[1])
        history = model.evaluate(val_images, val_masks, verbose=0)
        val_loss.append(history[0])
        val_acc.append(history[1])

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(range(1, 41), train_acc, label='Train Accuracy')
plt.plot(range(1, 41), val_acc, label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(range(1, 41), train_loss, label='Train Loss')
plt.plot(range(1, 41), val_loss, label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.show()
